In [ ]:
import os, shutil
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd

for folder in ["data_raw", "data_curated", "logs"]:
    os.makedirs(folder, exist_ok=True)

def run_ingest(source_name, input_file, required_cols, min_rows):
    now = datetime.now(ZoneInfo("Australia/Sydney"))
    run_id = now.strftime("%Y%m%d_%H%M%S")
    log = {"run_id": run_id,
           "timestamp": now.isoformat(timespec="seconds"),
           "source": source_name, "status": "started",
           "rows_in": 0, "rows_accepted": 0, "rows_flagged": 0, "error": ""}
    try:
        raw_path = f"data_raw/{source_name}_{run_id}.csv"
        if os.path.exists(raw_path):
            raise FileExistsError(f"{raw_path} already exists, refusing to overwrite")
        shutil.copy(input_file, raw_path)

        df = pd.read_csv(raw_path)
        log["rows_in"] = len(df)

        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Missing expected columns: {missing}")
        if len(df) < min_rows:
            raise ValueError(f"Only {len(df)} rows, expected at least {min_rows}")

        bad_mask = (df[required_cols].isna().any(axis=1)
                    | (df["Volume"] == 0)
                    | df["Date"].duplicated())
        bad, good = df[bad_mask], df[~bad_mask]
        bad.to_csv(f"logs/rejected_{source_name}_{run_id}.csv", index=False)
        good.to_csv(f"data_curated/{source_name}_{run_id}.csv", index=False)
        log["rows_accepted"], log["rows_flagged"] = len(good), len(bad)
        log["status"] = "success"

    except Exception as e:
        log["status"] = "failed"
        log["error"] = str(e)

    log_path = "logs/run_log.csv"
    pd.DataFrame([log]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
    print(log)

run_ingest("asx200", "asx200_historical.csv", ["Date", "Close", "Volume"], min_rows=100)

{'run_id': '20261023_192851', 'timestamp': '2026-10-23T19:28:51+11:00', 'source': 'asx200', 'status': 'success', 'rows_in': 255, 'rows_accepted': 253, 'rows_flagged': 2, 'error': ''}


In [ ]:
run_ingest("asx200", "asx200_historical.csv", ["Date", "Close", "Volume"], min_rows=100)
print(sorted(os.listdir("data_raw")))
pd.read_csv("logs/run_log.csv")

{'run_id': '20261023_193218', 'timestamp': '2026-10-23T19:32:18+11:00', 'source': 'asx200', 'status': 'success', 'rows_in': 255, 'rows_accepted': 253, 'rows_flagged': 2, 'error': ''}
['asx200_20261023_192851.csv', 'asx200_20261023_193218.csv']


,run_id,timestamp,source,status,rows_in,rows_accepted,rows_flagged,error
0,20261023_192851,2026-10-23T19:28:51+11:00,asx200,success,255,253,2,NaN
1,20261023_193218,2026-10-23T19:32:18+11:00,asx200,success,255,253,2,NaN


In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

Saving BUSA3022 Cash Rate Target 22Oct.xlsx to BUSA3022 Cash Rate Target 22Oct.xlsx
['BUSA3022 Cash Rate Target 22Oct.xlsx']


In [ ]:
RBA_FILE = "BUSA3022 Cash Rate Target 22Oct.xlsx"

sheets = pd.read_excel(RBA_FILE, sheet_name=None, header=None)
for name, sh in sheets.items():
    print("SHEET:", name, "| rows:", len(sh), "| columns:", sh.shape[1])
    print(sh.head(8))
    print()

SHEET: Sheet1 | rows: 347 | columns: 2
                                             0                         1
0  F1 INTEREST RATES AND YIELDS – MONEY MARKET                       NaN
1                                        Title          Cash Rate Target
2                                  Description  Cash Rate Target on date
3                                    Frequency                     Daily
4                                         Type                  Original
5                                        Units                  Per cent
6                                          NaN                       NaN
7                                          NaN                       NaN



In [ ]:
sh = sheets["Sheet1"]
print(sh.iloc[6:16])
print("...")
print(sh.tail(8))

                      0                    1
6                   NaN                  NaN
7                   NaN                  NaN
8                Source                  RBA
9      Publication date  2026-10-22 00:00:00
10            Series ID            FIRMMCRTD
11  2025-07-01 00:00:00                 3.85
12  2025-07-02 00:00:00                 3.85
13  2025-07-03 00:00:00                 3.85
14  2025-07-04 00:00:00                 3.85
15  2025-07-07 00:00:00                 3.85
...
                       0    1
339  2026-10-13 00:00:00  4.6
340  2026-10-14 00:00:00  4.6
341  2026-10-15 00:00:00  4.6
342  2026-10-16 00:00:00  4.6
343  2026-10-19 00:00:00  4.6
344  2026-10-20 00:00:00  4.6
345  2026-10-21 00:00:00  4.6
346  2026-10-22 00:00:00  NaN


In [ ]:
RBA_FILE = "BUSA3022 Cash Rate Target 22Oct.xlsx"
SERIES_ID = "FIRMMCRTD"

def run_rba_ingest(source_name, input_file, min_rows=100):
    now = datetime.now(ZoneInfo("Australia/Sydney"))
    run_id = now.strftime("%Y%m%d_%H%M%S")
    log = {"run_id": run_id, "timestamp": now.isoformat(timespec="seconds"),
           "source": source_name, "status": "started",
           "rows_in": 0, "rows_accepted": 0, "rows_flagged": 0, "error": ""}
    try:
        # 1. Save an untouched raw copy first (never overwrite)
        raw_path = f"data_raw/{source_name}_{run_id}.xlsx"
        if os.path.exists(raw_path):
            raise FileExistsError(f"{raw_path} already exists, refusing to overwrite")
        shutil.copy(input_file, raw_path)

        # 2. Read the raw copy and find the Series ID row
        sh = pd.read_excel(raw_path, header=None)
        id_rows = sh.index[sh[0].astype(str) == "Series ID"]
        if len(id_rows) == 0:
            raise ValueError("No 'Series ID' row found - possible schema change")
        hdr = id_rows[0]

        # 3. Check it is the series we expect
        found = str(sh.iloc[hdr, 1])
        if found != SERIES_ID:
            raise ValueError(f"Expected series {SERIES_ID} but found '{found}' - possible schema change")

        # 4. Take the date and rate columns below the header
        data = sh.iloc[hdr + 1:, [0, 1]].copy()
        data.columns = ["date", "rate"]
        log["rows_in"] = len(data)
        if len(data) < min_rows:
            raise ValueError(f"Only {len(data)} rows, expected at least {min_rows}")

        # 5. Flag bad rows: unreadable date, missing rate, duplicate date
        dates = pd.to_datetime(data["date"], errors="coerce")
        rates = pd.to_numeric(data["rate"], errors="coerce")
        bad_mask = dates.isna() | rates.isna() | dates.duplicated()
        data[bad_mask].to_csv(f"logs/rejected_{source_name}_{run_id}.csv", index=False)
        data[~bad_mask].to_csv(f"data_curated/{source_name}_{run_id}.csv", index=False)
        log["rows_accepted"], log["rows_flagged"] = int((~bad_mask).sum()), int(bad_mask.sum())

        # 6. Is the latest observation missing? Note it, don't assume why.
        if pd.isna(rates.iloc[-1]):
            log["error"] = f"WARNING: latest observation ({dates.iloc[-1].date()}) has no value - check publication timing"
        log["status"] = "success"

    except Exception as e:
        log["status"] = "failed"
        log["error"] = str(e)

    log_path = "logs/run_log.csv"
    pd.DataFrame([log]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
    print(log)

run_rba_ingest("rba_f1", RBA_FILE)
pd.read_csv("logs/run_log.csv").tail(3)

{'run_id': '20261023_210311', 'timestamp': '2026-10-23T21:03:11+11:00', 'source': 'rba_f1', 'status': 'success', 'rows_in': 336, 'rows_accepted': 335, 'rows_flagged': 1, 'error': 'WARNING: latest observation (2026-10-22) has no value - check publication timing'}


,run_id,timestamp,source,status,rows_in,rows_accepted,rows_flagged,error
0,20261023_192851,2026-10-23T19:28:51+11:00,asx200,success,255,253,2,NaN
1,20261023_193218,2026-10-23T19:32:18+11:00,asx200,success,255,253,2,NaN
2,20261023_210311,2026-10-23T21:03:11+11:00,rba_f1,success,336,335,1,WARNING: latest observation (2026-10-22) has n...


In [ ]:
# Make a broken copy of the RBA file: the Series ID is renamed, like a schema change
broken = pd.read_excel(RBA_FILE, header=None)
row = broken.index[broken[0].astype(str) == "Series ID"][0]
broken.iloc[row, 1] = "RENAMED_SERIES"
broken.to_excel("rba_broken.xlsx", header=False, index=False)

# Run the same pipeline on it
run_rba_ingest("rba_f1_broken", "rba_broken.xlsx")

# Show the evidence
print(sorted(os.listdir("data_raw")))
print(sorted(os.listdir("data_curated")))
pd.read_csv("logs/run_log.csv").tail(2)

{'run_id': '20261023_214740', 'timestamp': '2026-10-23T21:47:40+11:00', 'source': 'rba_f1_broken', 'status': 'failed', 'rows_in': 0, 'rows_accepted': 0, 'rows_flagged': 0, 'error': "Expected series FIRMMCRTD but found 'RENAMED_SERIES' - possible schema change"}
['asx200_20261023_192851.csv', 'asx200_20261023_193218.csv', 'rba_f1_20261023_210311.xlsx', 'rba_f1_broken_20261023_214740.xlsx']
['asx200_20261023_192851.csv', 'asx200_20261023_193218.csv', 'rba_f1_20261023_210311.csv']


,run_id,timestamp,source,status,rows_in,rows_accepted,rows_flagged,error
2,20261023_210311,2026-10-23T21:03:11+11:00,rba_f1,success,336,335,1,WARNING: latest observation (2026-10-22) has n...
3,20261023_214740,2026-10-23T21:47:40+11:00,rba_f1_broken,failed,0,0,0,Expected series FIRMMCRTD but found 'RENAMED_S...


AI LOG - entry 1
Date: 23 Oct 2026
What I asked: Claude wrote a script to save raw ASX data, validate it and write a run log.
What went wrong: The log timestamp said 08:05, but my clock said 7:05 PM. The code used datetime.now() with no time zone, so it logged Colab's server time (UTC) instead of Sydney time.
How I caught it: I compared the log timestamp with my own clock.
Fix: Changed datetime.now() to datetime.now(ZoneInfo("Australia/Sydney")). The new log shows 19:28:51+11:00.
Why it matters: The brief requires explicit time zone handling, and an ambiguous log time would break that.